In [1]:
!mamba install pandas

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 1.906 seconds
  Name    Version  Build                Channel
-------------------------------------------------------------
+ pandas  3.1.0    np23py313h124dbc7_0  emscripten-forge-4x
- pip     26.2.2   pyh145f28c_0         conda-forge


In [2]:
import pandas as pd
import numpy as np

In [3]:
orders_files = [
    "orders_2024_01.csv",
    "orders_2024_02.csv",
    "orders_2024_03.csv"
]

df_orders_raw = pd.concat(
    [pd.read_csv(file) for file in orders_files],
    ignore_index=True
)

print("Исходная информация о заказах:")
df_orders_raw.info(memory_usage="deep")

print("\nПервые 5 строк:")
display(df_orders_raw.head())

print("\nТипы данных:")
print(df_orders_raw.dtypes)

Исходная информация о заказах:
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        15000 non-null  int64  
 1   user_id         15000 non-null  int64  
 2   order_datetime  15000 non-null  str    
 3   product_id      15000 non-null  int64  
 4   price           15000 non-null  float64
 5   quantity        15000 non-null  int64  
 6   status          15000 non-null  str    
dtypes: float64(1), int64(4), str(2)
memory usage: 1.6 MB

Первые 5 строк:


,order_id,user_id,order_datetime,product_id,price,quantity,status
0,1,1044,2024-01-03 18:39:26,26,14038.15,1,refunded
1,2,1079,2024-01-03 17:45:43,16,9172.60,1,paid
2,3,1041,2024-01-06 22:46:38,25,9035.26,2,paid
3,4,1083,2024-01-08 21:03:51,9,15234.33,3,paid
4,5,1068,2024-01-03 23:26:53,24,4134.38,2,paid



Типы данных:
order_id            int64
user_id             int64
order_datetime        str
product_id          int64
price             float64
quantity            int64
status                str
dtype: object


In [5]:
orders_dtypes = {
    "order_id": "int32",
    "user_id": "int32",
    "product_id": "int16",
    "price": "float32",
    "quantity": "int16",
    "status": "category"
}

df_orders_opt = pd.concat(
    [
        pd.read_csv(
            file,
            dtype=orders_dtypes,
            parse_dates=["order_datetime"]
        )
        for file in orders_files
    ],
    ignore_index=True
)

print("Информация об оптимизированных заказах:")
df_orders_opt.info(memory_usage="deep")

print("\nПервые 5 строк:")
display(df_orders_opt.head())

print("\nОптимизированные типы данных:")
print(df_orders_opt.dtypes)

Информация об оптимизированных заказах:
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        15000 non-null  int32         
 1   user_id         15000 non-null  int32         
 2   order_datetime  15000 non-null  datetime64[us]
 3   product_id      15000 non-null  int16         
 4   price           15000 non-null  float32       
 5   quantity        15000 non-null  int16         
 6   status          15000 non-null  category      
dtypes: category(1), datetime64[us](1), float32(1), int16(2), int32(2)
memory usage: 366.4 KB

Первые 5 строк:


,order_id,user_id,order_datetime,product_id,price,quantity,status
0,1,1044,2024-01-03 18:39:26,26,14038.150391,1,refunded
1,2,1079,2024-01-03 17:45:43,16,9172.599609,1,paid
2,3,1041,2024-01-06 22:46:38,25,9035.259766,2,paid
3,4,1083,2024-01-08 21:03:51,9,15234.330078,3,paid
4,5,1068,2024-01-03 23:26:53,24,4134.379883,2,paid



Оптимизированные типы данных:
order_id                   int32
user_id                    int32
order_datetime    datetime64[us]
product_id                 int16
price                    float32
quantity                   int16
status                  category
dtype: object


In [6]:
memory_raw = (
    df_orders_raw.memory_usage(deep=True).sum() / 1024**2
)

memory_opt = (
    df_orders_opt.memory_usage(deep=True).sum() / 1024**2
)

memory_saved = memory_raw - memory_opt

saving_percent = (
    100 * (1 - memory_opt / memory_raw)
)

print(f"Память исходных данных: {memory_raw:.2f} MB")
print(f"Память оптимизированных данных: {memory_opt:.2f} MB")
print(f"Экономия памяти: {memory_saved:.2f} MB")
print(f"Процент экономии: {saving_percent:.2f}%")

Память исходных данных: 1.63 MB
Память оптимизированных данных: 0.36 MB
Экономия памяти: 1.27 MB
Процент экономии: 78.05%


In [7]:
memory_comparison = pd.DataFrame({
    "Тип данных": [
        "Исходные данные",
        "Оптимизированные данные"
    ],
    "Потребление памяти (MB)": [
        round(memory_raw, 2),
        round(memory_opt, 2)
    ]
})

display(memory_comparison)

print(f"Процент экономии памяти: {saving_percent:.2f}%")

,Тип данных,Потребление памяти (MB)
0,Исходные данные,1.63
1,Оптимизированные данные,0.36


Процент экономии памяти: 78.05%


In [8]:
memory_comparison.to_csv(
    "memory_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Таблица сохранена в memory_comparison.csv")

Таблица сохранена в memory_comparison.csv
